# 🤖 VGA: Vision-Geometry-Action Policy on LIBERO-Spatial
### End-to-End Training & Closed-Loop MuJoCo Simulation Pipeline on Kaggle GPU

This notebook is an end-to-end, single-click benchmark for training and evaluating the **Vision-Geometry-Action (VGA)** policy on the **LIBERO-Spatial** manipulation benchmark (Task 0: *pick up the black bowl between the plate and the ramekin and place it on the plate*).

---

### Key Architectural Specifications:
- **Vision Perception**: Pretrained `google/siglip-base-patch16-256` (frozen, 92.9M params)
- **Language & Multimodal Fusion**: Pretrained `HuggingFaceTB/SmolLM2-135M` (frozen, 134.5M params)
- **Trainable Action Head**: Space-to-Depth Projector + CentroidRayRoPE + 12-layer DiT Action Expert (36.3M params)
- **Parameter Ceiling**: **263.7M parameters** (strictly under the 0.5B ceiling)
- **Inference Speed**: **~7.7 ms / 16-step chunk (129.1 Hz)** on Tesla T4 Tensor Cores (exceeds the 50 Hz specification)
- **Camera Parity**: 180° camera rotation matching LeRobot's official `LiberoProcessorStep` convention

---

### Step-by-Step Execution Sequence:
1. **System & OpenGL Setup**: Install headless MuJoCo rendering and Python dependencies.
2. **Repository Synchronization**: Clone or pull the latest verified `vga-poc` code.
3. **Accelerator Diagnostics**: Verify GPU availability (Tesla T4 x 2 or P100) and VRAM.
4. **VGA Policy Training**: 10-shot demonstration fine-tuning (2,000 steps, ~10 min on 2x T4).
5. **Closed-Loop Simulation**: Interactive MuJoCo evaluation on LIBERO-Spatial Task 0.
6. **In-Notebook Video Replay**: Watch the recorded MP4 robot rollout directly in the cell output.
7. **Metrics Summary**: Inspect numerical benchmark metrics and telemetry.

## Step 1: Install System Libraries & Python Dependencies
Installs OSMesa / OpenGL headless drivers for MuJoCo simulation rendering, plus LeRobot, MuJoCo, and LIBERO.

In [1]:
# 1. System packages for headless OpenGL/EGL rendering in Kaggle Docker
!apt-get update -qq && apt-get install -y -qq libgl1-mesa-glx libosmesa6-dev

# 2. Python dependencies: PyAV video decoder, LeRobot, MuJoCo, and LIBERO simulation suite
!pip install --quiet "av<14" datasets num2words git+https://github.com/huggingface/lerobot.git zarr einops scipy torchvision peft accelerate matplotlib imageio imageio-ffmpeg "hf-libero>=0.1.4" mujoco

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libpciaccess-dev:amd64.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../00-libpciaccess-dev_0.16-3_amd64.deb ...
Unpacking libpciaccess-dev:amd64 (0.16-3) ...
Selecting previously unselected package libdrm-dev:amd64.
Preparing to unpack .../01-libdrm-dev_2.4.113-2~ubuntu0.22.04.1_amd64.deb ...
Unpacking libdrm-dev:amd64 (2.4.113-2~ubuntu0.22.04.1) ...
Preparing to unpack .../02-libegl-mesa0_23.2.1-1ubuntu3.1~22.04.4_amd64.deb ...
Unpacking libegl-mesa0:amd64 (23.2.1-1ubuntu3.1~22.04.4) over (23.2.1-1ubuntu3.1~22.04.3) ...
Preparing to unpack .../03-libgbm1_23.2.1-1ubuntu3.1~22.04.4_amd64.deb ...
Unpacking libgbm1:amd64 (23.2.1-1ubuntu3.1~22.04.4) over (23.2.1-1ubuntu3.1~22.04.3) ...
Preparing to unpack .../04-lib

## Step 2: Clone or Update the `vga-poc` Repository
Safely clones the repository or fetches the latest commit if the directory already exists.

In [11]:
import os

repo_dir = "/kaggle/working/vga-poc"
if not os.path.exists(repo_dir):
    print("Cloning vga-poc repository from GitHub...")
    !cd /kaggle/working && git clone https://github.com/Arkz-Deepak/vga-poc.git
else:
    print("Repository already exists. Updating to latest commit on main...")
    !cd /kaggle/working/vga-poc && git fetch origin && git reset --hard origin/main

print("Repository is synced and ready!")

Repository already exists. Updating to latest commit on main...
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 1.38 KiB | 472.00 KiB/s, done.
From https://github.com/Arkz-Deepak/vga-poc
   858b07a..a2a86a8  main       -> origin/main
HEAD is now at a2a86a8 fix(eval): add dynamic proximity grasp guard and settle dwell to prevent premature mid-air clamping
Repository is synced and ready!


In [13]:
    !cd /kaggle/working/vga-poc && git fetch origin && git reset --hard origin/main 

remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (7/7), done.
remote: Total 8 (delta 7), reused 7 (delta 7), pack-reused 1 (from 1)
Unpacking objects: 100% (8/8), 977 bytes | 244.00 KiB/s, done.
From https://github.com/Arkz-Deepak/vga-poc
   a2a86a8..8cf2c5d  main       -> origin/main
HEAD is now at 8cf2c5d fix(policy,eval): remove untrained prefix_proj noise injection and enforce descent to rim at 7.8cm


## Step 3: Hardware Diagnostics & Headless MuJoCo Configuration
Configures EGL headless GPU rendering and verifies CUDA device resources.

In [8]:
import os
import torch

# Configure headless MuJoCo rendering environment
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")

if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"Detected {num_gpus} GPU(s):")
    for i in range(num_gpus):
        name = torch.cuda.get_device_name(i)
        mem = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"  - GPU {i}: {name} ({mem:.2f} GB VRAM)")
else:
    print("⚠️ WARNING: Running on CPU! Enable GPU under 'Settings' -> 'Accelerator' (choose T4 x 2 or P100).")

PyTorch Version:  2.10.0+cu128
CUDA Available:   True
Detected 2 GPU(s):
  - GPU 0: Tesla T4 (14.56 GB VRAM)
  - GPU 1: Tesla T4 (14.56 GB VRAM)


## Step 4: Train the VGA Policy (10-Shot Demonstration Benchmark)
Fine-tunes the 36.3M parameter action head for 2,000 steps with AdamW and Cosine Annealing.
- **Frozen Backbones (227.4M params)**: Official `google/siglip-base-patch16-256` and `HuggingFaceTB/SmolLM2-135M`.
- **Trainable Action Head (36.3M params)**: Space-to-Depth Projector, CentroidRayRoPE, and 12-layer Action DiT.
- **Duration**: ~10 minutes on 2x Tesla T4 GPUs (or change `--steps 500` for a ~3.5 min smoke test).

In [9]:
# Run VGA policy training
!python /kaggle/working/vga-poc/scripts/train_vga.py \
    --shots 10 \
    --steps 2000 \
    --batch_size 8 \
    --lr 3e-4 \
    --output_dir /kaggle/working/vga-poc/checkpoints

      VGA PoC Training Run: 10-Shot Demo Benchmark       
Target Primary Device: cuda:0 (Total Available GPUs: 2)
  - GPU 0: Tesla T4 (14.56 GB VRAM)
  - GPU 1: Tesla T4 (14.56 GB VRAM)

--- 1. Dataset Ingestion & Demonstration Subsetting ---
Fetching 4 files: 100%|████████████████████████| 4/4 [00:00<00:00, 52103.16it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Filtered 10 episodes matching targets across 432 total episodes (skipped initial 0/task):
  - Episode  11 (84 frames): pick up the black bowl between the plate and the ramekin and place it 
  - Episode  12 (85 frames): pick up the black bowl between the plate and the ramekin and place it 
  - Episode  14 (128 frames): pick up the black bowl between the plate and the ramekin and place it 
  - Episode  21 (84 frames): pick up the black bowl between the plate and the ramekin and place it 
  - Episode  39 (102 frames): pick up the black bowl between the plate and the ramekin and place it 
  ... and 5 more episodes mat

## Step 5: Closed-Loop MuJoCo Simulation Evaluation
Evaluates the trained checkpoint on **LIBERO-Spatial Task 0**:
`"pick up the black bowl between the plate and the ramekin and place it on the plate"`

**Evaluation Features**:
- **Upright Camera Convention (`--flip_image`)**: Rotates OpenGL camera frames by 180° to match the Hugging Face / LeRobot dataset convention.
- **Pure Neural Policy Gripper**: Governed by the calibrated Schmitt trigger hysteresis (`schmitt_low=0.40, schmitt_high=0.60, min_hold_steps=60`) without hardcoded step overrides.
- **Video Recording**: High-fidelity 20 FPS MP4 rollouts saved to `/kaggle/working/vga-poc/results/videos/`.

In [14]:
# Run closed-loop MuJoCo simulation evaluation
!python /kaggle/working/vga-poc/scripts/eval_mujoco_closed_loop.py \
    --checkpoint /kaggle/working/vga-poc/checkpoints/vga_libero_10shot.pt \
    --num_episodes 3 \
    --max_steps 280 \
    --flip_image \
    --video_dir /kaggle/working/vga-poc/results/videos \
    --output_json /kaggle/working/vga-poc/results/closed_loop_simulation_results.json

   VGA Closed-Loop MuJoCo Simulation Benchmark (LIBERO-Spatial)   
Device: cuda
Loading checkpoint: /kaggle/working/vga-poc/checkpoints/vga_libero_10shot.pt
[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /usr/local/lib/python3.12/dist-packages/robosuite/scripts/setup_macros.py (__init__.py:9)
Loading weights: 100%|█| 208/208 [00:00<00:00, 1444.80it/s, Materializing param=
SiglipVisionModel LOAD REPORT from: google/siglip-base-patch16-256
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encod

## Step 6: Interactive Rollout Video Playback
Plays the recorded simulation video directly in the notebook.

In [15]:
import glob
import os
from IPython.display import HTML, display
from base64 import b64encode

video_files = sorted(glob.glob("/kaggle/working/vga-poc/results/videos/*.mp4"))

if video_files:
    print(f"Found {len(video_files)} video replay(s):")
    for vf in video_files:
        size_kb = os.path.getsize(vf) / 1024
        print(f"  - {os.path.basename(vf)} ({size_kb:.1f} KB)")
    
    # Pick the latest rollout video
    latest_video = video_files[-1]
    print(f"\nDisplaying: {os.path.basename(latest_video)}")
    
    with open(latest_video, "rb") as f:
        mp4_bytes = f.read()
    data_url = "data:video/mp4;base64," + b64encode(mp4_bytes).decode()
    
    display(HTML(f"""
    <div style="text-align: center; margin: 15px 0;">
        <h4 style="color: #2c3e50;">Simulation Rollout: {os.path.basename(latest_video)}</h4>
        <video width="640" height="640" controls autoplay loop style="border-radius: 8px; border: 2px solid #34495e; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
            <source src="{data_url}" type="video/mp4">
            Your browser does not support HTML5 video.
        </video>
    </div>
    """))
else:
    print("No MP4 simulation videos found in /kaggle/working/vga-poc/results/videos/")

Found 4 video replay(s):
  - task_0_ep_0_failed.mp4 (111.7 KB)
  - task_0_ep_1_failed.mp4 (114.3 KB)
  - task_0_ep_2_failed.mp4 (103.0 KB)
  - task_0_ep_2_success.mp4 (45.1 KB)

Displaying: task_0_ep_2_success.mp4


## Step 7: View Benchmark Metrics Summary
Displays quantitative simulation results recorded in JSON format.

In [ ]:
import json
import os

results_path = "/kaggle/working/vga-poc/results/closed_loop_simulation_results.json"
if os.path.exists(results_path):
    with open(results_path, "r") as f:
        metrics = json.load(f)
    print(json.dumps(metrics, indent=2))
else:
    print(f"Results file not found at: {results_path}")